# Saving Models

In [1]:
from ngboost import NGBRegressor

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

X, Y = load_diabetes(return_X_y=True)
X_reg_train, X_reg_test, Y_reg_train, Y_reg_test = train_test_split(X, Y, test_size=0.2)

Save ngboost models with [skops](https://skops.readthedocs.io). Install it with `pip install skops`.

Do not save models with `pickle` or `joblib`. When you load a pickle file, it can run any code that it holds. Also, ngboost does not keep pickle files from one version working in the next.

Fit a model:

In [2]:
ngb = NGBRegressor().fit(X_reg_train, Y_reg_train)

[iter 0] loss=5.7528 val_loss=0.0000 scale=1.0000 norm=65.0676


[iter 100] loss=5.4136 val_loss=0.0000 scale=2.0000 norm=91.9029


[iter 200] loss=5.1630 val_loss=0.0000 scale=1.0000 norm=36.5190


[iter 300] loss=5.0231 val_loss=0.0000 scale=2.0000 norm=65.8986


[iter 400] loss=4.9137 val_loss=0.0000 scale=1.0000 norm=30.6910


Save it to a skops file. Compress the file to make it smaller:

In [3]:
from zipfile import ZIP_DEFLATED

import skops.io as sio

sio.dump(ngb, "ngbtest.skops", compression=ZIP_DEFLATED)

skops loads only the types that you trust. Do not trust all the types that a file asks for. Make a list of the types that you expect, and compare the file with it. For this model, the list has the model class, the distribution, the scoring rule and the scikit-learn tree:

In [4]:
trusted = [
    "ngboost.api.NGBRegressor",
    "ngboost.distns.normal.Normal",
    "ngboost.scores.LogScore",
    "sklearn.tree._tree.Tree",
]

file_types = sio.get_untrusted_types(file="ngbtest.skops")
assert set(file_types) <= set(trusted), file_types
file_types

['ngboost.api.NGBRegressor',
 'ngboost.distns.normal.Normal',
 'ngboost.scores.LogScore',
 'sklearn.tree._tree.Tree']

Load the model with that list:

In [5]:
ngb_skops = sio.load("ngbtest.skops", trusted=trusted)

ngb_skops.pred_dist(X_reg_test)[0:5].params

{'loc': array([257.15247606, 165.69544341, 253.12377   ,  85.9201257 ,
         73.62197247]),
 'scale': array([17.67561646, 36.25707834, 11.50566048, 32.8306402 , 29.45793699])}

skops does not trust `sklearn.tree._tree.Tree` by default. It does not check the nodes of a saved tree, and a tree with bad nodes can crash Python when the model predicts. If you do not trust the person who made a file, load it only in a process where a crash does no damage.